In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import datetime
import pytz

NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../")

In [2]:
from MDP.IRSwaptions.IRSwaptionMDP import IRSwaptionMDP
from Query.Base.query_resolution import resolve_query
from Query.IRSwaptions import IRSwaptionQuery, IRSwaptionStructure, IRSwaptionValue


In [3]:
# mdp = IRSwaptionMDP(
#     source="GSQUANT-QL",
#     curve_source="ERIS_EOD_LIVE-QL_BASIC",
# )

# mdp = IRSwaptionMDP(
#     source="MONKEYCUBE-QL",
#     curve_source="ERIS_EOD_LIVE-QL_BASIC",
#     data_dir=r'C:\Users\chris\clee\ARBS\MDP\IRSwaptions\MONKEYCUBE\YCMONKEY_USD_VOL_CUBE_GAMMA_MIX'
# )

mdp = IRSwaptionMDP(
    source="GSQUANT_MC_ENHANCED-QL",
    curve_source="ERIS_EOD_LIVE-QL_BASIC",
    data_dir=r'C:\Users\chris\clee\ARBS\MDP\IRSwaptions\MONKEYCUBE\YCMONKEY_USD_VOL_CUBE_GAMMA_MIX'
)

In [5]:
as_of = datetime.date(2026, 3, 13)
ctx = mdp.get_pricer(
    {
        "curve_name": "USD-SOFR-1D",
        "timestamp": as_of,
        "ignore_cache": True,
    }
)
ctx

IRSwaptionMarketContext(curve_name='USD-SOFR-1D', as_of_date=datetime.date(2026, 3, 13), curve=QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x000001AE162E84B0> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x000001AE15F47F70> >, _meta_data={'timestamp': datetime.date(2026, 3, 13)}), curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x000001AE162E84B0> >, swap_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x000001AE15F47F70> >, vol_handle=<QuantLib.QuantLib.SwaptionVolatilityStructureHandle; proxy of <Swig Object of type 'Handle< SwaptionVolatilityStructure > *' at 0x000001AE15577930> >, pricing_engine=<QuantLib.QuantLib.BachelierSwaptionEngine; proxy of <Swig Object of type 'ext::shared_

In [22]:
q = IRSwaptionQuery(
	shorthand="1m7y",
	structure=IRSwaptionStructure.PAYER,
	# structure_kwargs={"premium_bps": 62.25 }
)

q_eff = resolve_query(q, timestamp=as_of, pricer_or_curve=ctx)
package, weights = q_eff.resolve_package(pricer_or_curve=ctx)
vmap = q_eff.build_value_map(pricer_or_curve=ctx, package=package, risk_weights=weights)
float(vmap.apply(IRSwaptionValue.NVOL)), float(vmap.apply(IRSwaptionValue.FWD_PREM))

(89.88413819350438, 64.48745122259899)

In [14]:
q = IRSwaptionQuery(
	shorthand="1y1y",
    # strike="ATMF",
    strike="ATMF+200",
    structure_kwargs={"notional": 100_000_000}
)

q_eff = resolve_query(q, timestamp=as_of, pricer_or_curve=ctx)
package, weights = q_eff.resolve_package(pricer_or_curve=ctx)
vmap = q_eff.build_value_map(pricer_or_curve=ctx, package=package, risk_weights=weights)
float(vmap.apply(IRSwaptionValue.NVOL)), float(vmap.apply(IRSwaptionValue.FWD_PREM)), float(vmap.apply(IRSwaptionValue.VEGA_01))

(260.70998625020206, 33.24116440883732, 2856.9753722339487)

In [15]:
q = IRSwaptionQuery(
	shorthand="1y1y",
    strike="ATMF-100",
)

q_eff = resolve_query(q, timestamp=as_of, pricer_or_curve=ctx)
package, weights = q_eff.resolve_package(pricer_or_curve=ctx)
vmap = q_eff.build_value_map(pricer_or_curve=ctx, package=package, risk_weights=weights)
float(vmap.apply(IRSwaptionValue.NVOL)), float(vmap.apply(IRSwaptionValue.FWD_PREM))

(103.03660408100285, 9.150862996426993)

In [51]:
from MDP.STIRCapFloors.STIRCapFloorMDP import STIRCapFloorMDP
from Query.STIRCapFloors.STIRCapFloorQuery import STIRCapFloorQuery
from Query.STIRCapFloors.STIRCapFloorStructure import STIRCapFloorStructure
from Query.STIRCapFloors.STIRCapFloorValue import STIRCapFloorValue

capfloor_mdp = STIRCapFloorMDP(
    curve_source="ERIS_EOD_LIVE-QL_BASIC",
    option_source="BARCHART_STIRFO-QL",
)

q = STIRCapFloorQuery(
    structure=STIRCapFloorStructure.CAP,
    shorthand="1Yx2Y",
    weight_method="duration",
    strike_convention="atm_per_caplet",
)

ctx = capfloor_mdp.get_pricer(q.build_mdp_request(datetime.date(2026, 3, 10)))
ctx

STIRCapFloorMarketContext(structure='CAP', curve_name='USD-SOFR-1D', as_of_date=datetime.date(2026, 3, 10), swap_start=datetime.date(2027, 3, 17), swap_end=datetime.date(2029, 3, 21), weight_method='duration', strike_convention='atm_per_caplet', contracts=1.0, curve=QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x0000018DBB0080F0> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x0000018DBB008130> >, _meta_data={'timestamp': datetime.date(2026, 3, 10)}), legs=(STIRCapFloorLegMarket(option_symbol='SFRH27|9675P', requested_symbol='SFRH27|ATMP', right='P', underlying_contract='SFRH27', reference_quarter_start=datetime.date(2027, 3, 17), reference_quarter_end=datetime.date(2027, 6, 16), strike_price=96.75, strike_rate=3.25, requested_strike_price=96.75, requested_strike_rate=3.25, economic_weight=0.1273674442916

In [52]:
ctx.id()
ctx.meta()["strip_contracts"]

# for leg in ctx.legs:
#     print(
#         leg.underlying_contract,
#         leg.option_symbol,
#         leg.requested_symbol,
#         leg.quote_source,
#         leg.economic_weight,
#     )

['SFRH27',
 'SFRM27',
 'SFRU27',
 'SFRZ27',
 'SFRH28',
 'SFRM28',
 'SFRU28',
 'SFRZ28']